# 预测
不折腾了，直接采用基线模型预测，然后提交。首先是数据集和导入，然后是训练，然后是预测。预测的时候也要预先处理数据，并且不能删列，还要对齐索引

In [18]:
from pathlib import Path

import pandas as pd
from sklearn.base import BaseEstimator

CURRENT_DIR = Path().cwd()
if CURRENT_DIR.name == "notebooks":
    #当前在notebooks/下面
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    #当前在根目录
    PROJECT_ROOT = CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

df = pd.read_csv(PROCESSED_DIR / "train_cleaned.csv")
df["MSSubClass"] = df["MSSubClass"].astype(str)


In [19]:
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import numpy as np

# 数据
X = df.drop(columns="SalePrice")
y = df["SalePrice"]
num_cols = X.select_dtypes("number").columns
cat_cols = X.select_dtypes("str").columns

# 处理器定义
preprocessor = ColumnTransformer(
    transformers=[
        ("num", RobustScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore",
                              drop="first",
                              sparse_output=False),
         cat_cols),
    ]
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("LRmodel", LinearRegression()),
    ]
)

# 测试pipeline
# 数据应该是刚刚从df里面分出来的X和y，y提前取log就行了，其他的不要做，不能重复处理
pipeline.fit(X, np.log(y))
print("基线模型训练完成.") # 加一个防止输出html进而导致渲染错误

基线模型训练完成.


接下来要额外处理一下test数据，test数据有额外的缺失值：
```
Exterior2nd     0.000685
Exterior1st     0.000685
BsmtFinSF2      0.000685
BsmtFinSF1      0.000685
BsmtUnfSF       0.000685
TotalBsmtSF     0.000685
SaleType        0.000685
KitchenQual     0.000685
GarageArea      0.000685
GarageCars      0.000685
Functional      0.001371
Utilities       0.001371
BsmtFullBath    0.001371
BsmtHalfBath    0.001371
MSZoning        0.002742
```

当然，清理前务必提前保存testID，因为清理会把这列数据直接删掉

In [20]:
from src.cleaning import clean_data
df_test = pd.read_csv(RAW_DIR / "test.csv")
df_test_Id = df_test["Id"].copy()
df_test_Id.to_csv(PROCESSED_DIR / "test_Id.csv", index=False)

df_test = clean_data(df_test, True)

错误列名矫正完成.
存在缺失值NA的列:
Exterior2nd     0.000685
Exterior1st     0.000685
BsmtUnfSF       0.000685
BsmtFinSF2      0.000685
BsmtFinSF1      0.000685
GarageArea      0.000685
KitchenQual     0.000685
TotalBsmtSF     0.000685
GarageCars      0.000685
SaleType        0.000685
BsmtHalfBath    0.001371
BsmtFullBath    0.001371
Utilities       0.001371
Functional      0.001371
MSZoning        0.002742
MasVnrArea      0.010281
BsmtFinType1    0.028787
BsmtFinType2    0.028787
BsmtQual        0.030158
BsmtExposure    0.030158
BsmtCond        0.030843
GarageType      0.052090
GarageFinish    0.053461
GarageCond      0.053461
GarageQual      0.053461
GarageYrBlt     0.053461
LotFrontage     0.155586
FireplaceQu     0.500343
MasVnrType      0.612748
Fence           0.801234
Alley           0.926662
MiscFeature     0.965045
PoolQC          0.997944
缺失值清理完成.
清理一次之后，存在缺失值NA的列:
Exterior2nd     0.000685
Exterior1st     0.000685
BsmtFinSF2      0.000685
BsmtFinSF1      0.000685
BsmtUnfSF       0.000685
To

接下来看一下

In [21]:
df_test.columns

Index(['MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street', 'Alley',
       'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope',
       'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle',
       'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'RoofStyle',
       'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'MasVnrArea',
       'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond',
       'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1', 'BsmtFinType2',
       'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating', 'HeatingQC',
       'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF',
       'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath',
       'Bedroom', 'Kitchen', 'KitchenQual', 'TotRmsAbvGrd', 'Functional',
       'Fireplaces', 'FireplaceQu', 'GarageType', 'GarageYrBlt',
       'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual', 'GarageCond',
       'PavedDrive', 'WoodDeckSF',

然后test也需要预处理。我们已经写好了函数，并固定下来了

In [22]:
from src.preprocessing import build_preprocessor
preprocessor = build_preprocessor()
pipeline_baseline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("LRmodel", LinearRegression())
    ]
)
pipeline_baseline.fit(X, np.log(y))
pred_log = pipeline_baseline.predict(df_test)
pred_price = np.exp(pred_log)

E:\Software\PythonProjects\Housing_Prices\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


接下来生成submission

In [23]:
submission = pd.DataFrame(
    {
        "Id":df_test_Id,
        "SalePrice": pred_price
    }
)
submission.to_csv(PROCESSED_DIR / "submission.csv", index=False)

这样，就生成，完成了